In [3]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [4]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [5]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [6]:
# Read all images in the images folder, feed each into Claude
import os

image_dir = "images"

for filename in sorted(os.listdir(image_dir)):
    if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".gif", ".webp")):
        continue

    filepath = os.path.join(image_dir, filename)
    media_type = "image/png" if filename.lower().endswith(".png") else "image/jpeg"

    with open(filepath, "rb") as f:
        image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

    messages = []
    add_user_message(messages, [
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": media_type,
                "data": image_bytes,
            },
        },
        {
            "type": "text",
            "text": prompt,
        },
    ])

    response = chat(messages)
    print(f"=== {filename} ===")
    print(text_from_message(response))
    print()

=== prop1.png ===
# Satellite Image Fire Risk Analysis

**1. Residence Identification:**
The primary residence is located in the center-left portion of the image, appearing as a light gray/tan rectangular structure with a clearly defined roof, surrounded by dense vegetation on all sides with what appears to be a cleared area or driveway approach from the left side.

**2. Tree Overhang Analysis:**
Multiple large tree canopies directly overhang the residence roof, with dense dark green foliage covering approximately 50-75% of the visible roof surface, particularly concentrated on the northern and eastern portions of the structure.

**3. Fire Risk Assessment:**
The overhanging trees create multiple ember catch points and establish direct fuel continuity between the surrounding forest canopy and the structure, with branches appearing to extend completely over the roof in several areas, creating significant wildfire vulnerability with no apparent defensible breaks.

**4. Defensible Space Id